In [1]:
import pandas as pd
import os
import dask.dataframe as dd

In [2]:
d_path = "/projects/p32795/weijian/tmps/03_02_xgb_095_vnv"
filename = "field_451_vs.csv"

# read the csv file
df = pd.read_csv(os.path.join(d_path, filename))

# print the first 5 rows of the dataframe
print(df.head())



   Unnamed: 0             _id       Gaia_EDR3___id        AllWISE___id  \
0          12  10451361000034  2520035850059265792  349106001351037504   
1          27  10451361000067  2520035643900835840  349106001351037440   
2         147  10451361000449  2520032826402290560  349106001351037568   
3         149  10451361000453  2520032822107331840  349106001351037504   
4         179  10451361000543  2520032414085430272  349106001351037504   

         PS1_DR1___id         ra       dec    period  field   ccd  ...  \
0  115620342109151952  34.210875  6.351220  0.073873  451.0  10.0  ...   
1  115610342213992400  34.221329  6.343301  0.031009  451.0  10.0  ...   
2  115500342353436096  34.235330  6.254701  1.527467  451.0  10.0  ...   
3  115500342317575472  34.231764  6.254179  0.054275  451.0  10.0  ...   
4  115480342092743920  34.209261  6.236222  0.022343  451.0  10.0  ...   

   hp_xgb  i_xgb  blyr_xgb  yso_xgb  saw_xgb  ceph2_xgb  rrab_xgb  sin_xgb  \
0     0.0    0.0       0.0      

In [5]:
df.iloc[11500:11501][['_id','ra', 'dec', 'ccd', 'quad']]

,_id,ra,dec,ccd,quad
11500,10451631002595,30.49325,6.925177,16.0,4.0


In [5]:
df.columns

Index(['Unnamed: 0', '_id', 'Gaia_EDR3___id', 'AllWISE___id', 'PS1_DR1___id',
       'ra', 'dec', 'period', 'field', 'ccd', 'quad', 'filter', 'sin_dnn',
       'rscvn_dnn', 'emsms_dnn', 'ea_dnn', 'rrlyr_dnn', 'el_dnn', 'saw_dnn',
       'bogus_dnn', 'wuma_dnn', 'e_dnn', 'yso_dnn', 'ceph_dnn', 'cv_dnn',
       'srv_dnn', 'fla_dnn', 'blyr_dnn', 'wp_dnn', 'eb_dnn', 'blher_dnn',
       'hp_dnn', 'ext_dnn', 'dip_dnn', 'rrab_dnn', 'vnv_dnn', 'pnp_dnn',
       'wvir_dnn', 'dp_dnn', 'bright_dnn', 'ceph2_dnn', 'dscu_dnn', 'bis_dnn',
       'osarg_dnn', 'rrblz_dnn', 'i_dnn', 'puls_dnn', 'mp_dnn', 'rrc_dnn',
       'mir_dnn', 'agn_dnn', 'longt_dnn', 'blend_dnn', 'ew_dnn', 'lpv_dnn',
       'rrd_dnn', 'rrd_xgb', 'wvir_xgb', 'dp_xgb', 'blher_xgb', 'lpv_xgb',
       'agn_xgb', 'dscu_xgb', 'puls_xgb', 'rrc_xgb', 'mir_xgb', 'e_xgb',
       'rscvn_xgb', 'rrlyr_xgb', 'bis_xgb', 'emsms_xgb', 'mp_xgb', 'ew_xgb',
       'bogus_xgb', 'ea_xgb', 'vnv_xgb', 'pnp_xgb', 'longt_xgb', 'dip_xgb',
       'cv_xgb', '

In [9]:
ztf_data = "/projects/b1094/stroh/software/catalogs/ztf/lc_dr23/0/field000451/ztf_000451_zr_c16_q4_dr23.parquet"
ddf = dd.read_parquet(ztf_data, engine="pyarrow")

In [30]:
from astropy.coordinates import SkyCoord
from astropy import units as u
from astropy.table import Table

stars_df = pd.read_csv(os.path.join(d_path, filename))
group_df = stars_df[(stars_df['ccd'] == 16.0) & (stars_df['quad'] == 4.0)]
# group_df = Table.from_pandas(group_df)
group_coords = SkyCoord(ra=group_df['ra'].values*u.degree, 
                                dec=group_df['dec'].values*u.degree)

# Read parquet file (once for all stars in this group)
df = pd.read_parquet(ztf_data)
df = Table.from_pandas(df)

# Create catalog coordinates once
catalog_coords = SkyCoord(ra=df['objra']*u.degree, 
                        dec=df['objdec']*u.degree)

# Match all stars in this group to the catalog at once
idx, sep2d, _ = group_coords.match_to_catalog_sky(catalog_coords)

In [31]:
within_radius = sep2d < (2.0 * u.arcsec)

In [33]:
group_df.iloc[0:1]

,Unnamed: 0,_id,Gaia_EDR3___id,AllWISE___id,PS1_DR1___id,ra,dec,period,field,ccd,...,hp_xgb,i_xgb,blyr_xgb,yso_xgb,saw_xgb,ceph2_xgb,rrab_xgb,sin_xgb,osarg_xgb,rrblz_xgb
11500,280065,10451631002595,2567768536039114880,305107501351004992,116310304932790672,30.49325,6.925177,0.132373,451.0,16.0,...,0.0,0.0,0.0,0.0,0.0,0.23,0.0,0.0,0.0,0.33


In [38]:
group_df.iloc[22:23]

,Unnamed: 0,_id,Gaia_EDR3___id,AllWISE___id,PS1_DR1___id,ra,dec,period,field,ccd,...,hp_xgb,i_xgb,blyr_xgb,yso_xgb,saw_xgb,ceph2_xgb,rrab_xgb,sin_xgb,osarg_xgb,rrblz_xgb
11522,286568,10451633000001,2567824331958920704,305107501351017216,116870306507939984,30.650783,7.399586,0.214625,451.0,16.0,...,0.0,0.0,0.0,0.0,0.0,0.23,0.0,0.0,0.0,0.33


In [34]:
sep2d[0]

<Angle 8.73307057e-06 deg>

In [35]:
sep2d[0] < 2.0 * u.arcsec

True

In [37]:
# find the index of false in within_radius
import numpy as np
np.where(~within_radius)[0]

array([ 22,  23,  24,  25,  26,  61, 118, 184, 209, 211, 216, 255, 283,
       316, 317, 318, 319, 336, 395, 405, 423, 426, 447, 463, 467, 469,
       470, 484, 486])

In [41]:
sep2d[22]

<Angle 0.0123502 deg>

In [15]:
len(sep2d)

489

In [13]:
df['quad']

0       1.0
1       1.0
2       1.0
3       1.0
4       1.0
       ... 
7995    4.0
7996    4.0
7997    4.0
7998    4.0
7999    4.0
Name: quad, Length: 8000, dtype: float64

In [4]:
ddf.head()

,objectid,filterid,fieldid,rcid,objra,objdec,nepochs,hmjd,mag,magerr,clrcoeff,catflags
0,249210100000000,2,249,36,32.267223,-22.428663,156,"[58338.50301, 58342.50562, 58350.51245, 58356....","[18.17188, 18.196068, 18.194172, 18.19894, 18....","[0.047784667, 0.048660453, 0.04859116, 0.04876...","[0.09467691, 0.11537975, 0.097610936, 0.098518...","[0, 0, 0, 0, 0, 0, 0, 0, 32768, 0, 0, 0, 0, 0,..."
1,249210100000001,2,249,36,32.540653,-22.425495,124,"[58338.50299, 58342.5056, 58350.51242, 58356.4...","[19.206804, 19.216352, 19.249777, 19.302952, 1...","[0.10516854, 0.105912715, 0.108549625, 0.11284...","[0.09467691, 0.11537975, 0.097610936, 0.098518...","[0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 655..."
3,249210100000003,2,249,36,32.496693,-22.430059,46,"[58356.44154, 58374.48519, 58377.40263, 58397....","[20.092653, 20.37327, 20.434294, 20.741966, 20...","[0.1853823, 0.20604776, 0.2088704, 0.23000813,...","[0.09851892, 0.10393803, 0.11349469, 0.1028326...","[0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, ..."
4,249210100000004,2,249,36,31.959435,-22.436485,168,"[58338.50304, 58342.50565, 58350.51247, 58356....","[14.649779, 14.646492, 14.663121, 14.636834, 1...","[0.0148298815, 0.014833384, 0.014815826, 0.014...","[0.09467691, 0.11537975, 0.097610936, 0.098518...","[0, 0, 0, 0, 0, 0, 0, 0, 32768, 32768, 0, 0, 0..."
5,249210100000005,2,249,36,32.633781,-22.428940,169,"[58338.50298, 58342.50559, 58350.51242, 58356....","[14.878163, 14.88268, 14.897228, 14.851554, 14...","[0.014630109, 0.014627093, 0.014617645, 0.0146...","[0.09467691, 0.11537975, 0.097610936, 0.098518...","[0, 0, 0, 0, 0, 0, 0, 0, 0, 32768, 32768, 0, 0..."


In [5]:
ddf.dtypes

objectid      int64
filterid       int8
fieldid       int16
rcid           int8
objra       float32
objdec      float32
nepochs       int64
hmjd         object
mag          object
magerr       object
clrcoeff     object
catflags     object
dtype: object

In [9]:
ddf.columns

Index(['objectid', 'filterid', 'fieldid', 'rcid', 'objra', 'objdec', 'nepochs',
       'hmjd', 'mag', 'magerr', 'clrcoeff', 'catflags'],
      dtype='object')

In [23]:
len(str(a))

14

In [40]:
a = df["_id"].iloc[0]
print(a)
# find a in ddf["objectid"]
b = ddf[ddf["objectid"] == f'{a}']
b.head()

10249361000005


,objectid,filterid,fieldid,rcid,objra,objdec,nepochs,hmjd,mag,magerr,clrcoeff,catflags


In [41]:
b.compute()

,objectid,filterid,fieldid,rcid,objra,objdec,nepochs,hmjd,mag,magerr,clrcoeff,catflags


In [24]:
df[df["_id"] == str(a)][['ccd', 'quad']]

,ccd,quad


In [17]:
df.head(2)

,Unnamed: 0,_id,Gaia_EDR3___id,AllWISE___id,PS1_DR1___id,ra,dec,period,field,ccd,...,hp_xgb,i_xgb,blyr_xgb,yso_xgb,saw_xgb,ceph2_xgb,rrab_xgb,sin_xgb,osarg_xgb,rrblz_xgb
0,0,10249361000005,5124074212386822528,325022801351040064,81040319672163360,31.967209,-22.464097,454.943205,249.0,10.0,...,0.00,0.17,0.0,0.0,0.0,0.23,0.00,0.02,0.0,0.33
1,37,10249361000205,5124037310028047616,325022801351045312,80990326464831024,32.646504,-22.507725,868.527936,249.0,10.0,...,0.01,0.02,0.0,0.0,0.0,0.24,0.01,0.00,0.0,0.33


In [33]:
# get the first objectid
ddf['objectid'][0]

# get the value from <dask_expr.expr.Scalar: expr=ReadParquetFSSpec(76fed90)['objectid'][0], dtype=int64>
objectid = ddf['objectid'][0]


<dask_expr.expr.Scalar: expr=ReadParquetFSSpec(76fed90)['objectid'][0], dtype=int64>

In [35]:
c = objectid.compute()

In [38]:
c

249210100000000

In [37]:
len(str(c))

15

In [29]:
len(str(objectid))

84

In [30]:
str(objectid)

"<dask_expr.expr.Scalar: expr=ReadParquetFSSpec(76fed90)['objectid'][0], dtype=int64>"

In [42]:
import datasets

d = datasets.load_from_disk("/projects/p32795/weijian/queried_scope_from_ztf/ztf_matches_dataset")

/projects/p32015/conda-envs/pretrain/lib/python3.11/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [55]:
d[200]['band']

'g'

In [46]:
len(d)

1401899

In [53]:
import re
filename = "ztf_000451_zg_c16_q4_dr23.parquet"
match = re.search(r'_z([a-z])_', filename)
match.group(1)

'g'